In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/habibamuhammed1/streamlitarabic/arabic_sentiment (1).keras


In [2]:
!pip install -q streamlit openai-whisper pyngrok
!apt-get update -qq
!apt-get install -y -qq ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.0 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 71.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 90.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 6.8 MB/s eta 0:00:00:00:0100:01
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
%%writefile /kaggle/working/app.py

import streamlit as st
import tensorflow as tf
import whisper
import numpy as np
import tempfile
import os
import re
import glob


# =========================================================
# PAGE CONFIG
# =========================================================

st.set_page_config(
    page_title="Arabic Sentiment Analysis",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="expanded"
)


# =========================================================
# CUSTOM CSS
# =========================================================

st.markdown("""
<style>

.stApp {
    background-color: #08090b;
    color: white;
}

#MainMenu {visibility: hidden;}
footer {visibility: hidden;}
header {visibility: hidden;}


/* Sidebar */

[data-testid="stSidebar"] {
    background-color: #101114;
    border-right: 1px solid #242424;
}


/* Main Container */

.block-container {
    padding-top: 3rem;
    padding-left: 5rem;
    padding-right: 5rem;
}


/* Main Title */

.main-title {
    font-size: 50px;
    font-weight: 800;
    color: #f5f5f5;
    margin-bottom: 10px;
}


/* Subtitle */

.subtitle {
    font-size: 18px;
    color: #aaaaaa;
    margin-bottom: 35px;
}


/* Section Title */

.section-title {
    font-size: 28px;
    font-weight: 700;
    margin-top: 25px;
    margin-bottom: 20px;
}


/* Text Area */

textarea {
    background-color: #24262b !important;
    color: white !important;
    border-radius: 15px !important;
    border: 1px solid #555 !important;
}


/* Buttons */

.stButton > button {
    background-color: #1c1d21;
    color: white;
    border: none;
    border-radius: 25px;
    padding: 12px 28px;
    font-weight: 600;
}

.stButton > button:hover {
    background-color: #e31b23;
    color: white;
}


/* Metrics */

[data-testid="stMetric"] {
    background-color: #15161a;
    padding: 18px;
    border-radius: 15px;
    border: 1px solid #28292e;
}


/* Tabs */

.stTabs [data-baseweb="tab-list"] {
    gap: 30px;
}

.stTabs [data-baseweb="tab"] {
    color: #999;
    font-size: 16px;
}

.stTabs [aria-selected="true"] {
    color: #ff4040 !important;
    border-bottom-color: #ff4040 !important;
}

</style>
""", unsafe_allow_html=True)


# =========================================================
# FIND KERAS MODEL AUTOMATICALLY
# =========================================================

def find_model():

    possible_paths = []

    possible_paths += glob.glob("/kaggle/working/*.keras")

    possible_paths += glob.glob(
        "/kaggle/input/**/*.keras",
        recursive=True
    )

    if possible_paths:
        return possible_paths[0]

    return None


MODEL_PATH = find_model()


# =========================================================
# LOAD SENTIMENT MODEL
# =========================================================

@st.cache_resource
def load_sentiment_model(model_path):

    return tf.keras.models.load_model(
        model_path,
        compile=False
    )


# =========================================================
# LOAD WHISPER
# =========================================================

@st.cache_resource
def load_whisper_model():

    return whisper.load_model("base")


# =========================================================
# CLEAN ARABIC TEXT
# =========================================================

def clean_text(text):

    text = str(text)

    # Remove Arabic diacritics
    text = re.sub(
        r'[\u0617-\u061A\u064B-\u0652\u0640]',
        '',
        text
    )

    # Normalize Arabic letters
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# =========================================================
# PREDICT SENTIMENT
# =========================================================

def predict_sentiment(text, model):

    cleaned_text = clean_text(text)

    prediction = model.predict(
        tf.constant([cleaned_text], dtype=tf.string),
        verbose=0
    )

    positive_probability = float(prediction[0][0])

    # Safety
    positive_probability = max(
        0,
        min(1, positive_probability)
    )

    negative_probability = 1 - positive_probability


    # Classification threshold
    THRESHOLD = 0.50

    if positive_probability >= THRESHOLD:

        sentiment = "Positive 😊"

    else:

        sentiment = "Negative 😞"


    return {
        "cleaned_text": cleaned_text,
        "positive": positive_probability,
        "negative": negative_probability,
        "prediction": sentiment
    }


# =========================================================
# TRANSCRIBE AUDIO / VIDEO
# =========================================================

def transcribe_media(file_path):

    whisper_model = load_whisper_model()

    result = whisper_model.transcribe(
        file_path,
        language="ar",
        fp16=False
    )

    return result["text"]


# =========================================================
# DISPLAY RESULT
# =========================================================

def display_result(result):

    st.markdown("---")

    st.markdown(
        '<div class="section-title">📊 Analysis Result</div>',
        unsafe_allow_html=True
    )


    if "Positive" in result["prediction"]:

        st.success(
            "😊 POSITIVE SENTIMENT DETECTED"
        )

    else:

        st.error(
            "😞 NEGATIVE SENTIMENT DETECTED"
        )


    col1, col2, col3 = st.columns(3)


    col1.metric(
        "😊 Positive Probability",
        f"{result['positive'] * 100:.2f}%"
    )


    col2.metric(
        "😞 Negative Probability",
        f"{result['negative'] * 100:.2f}%"
    )


    col3.metric(
        "🎯 Prediction",
        result["prediction"]
    )


# =========================================================
# CHECK MODEL
# =========================================================

if MODEL_PATH is None:

    st.error("❌ No .keras model found!")

    st.stop()


model = load_sentiment_model(MODEL_PATH)


# =========================================================
# SIDEBAR
# =========================================================

with st.sidebar:

    st.markdown("# 🧠")

    st.markdown("## AI Sentiment")

    st.caption("Arabic NLP System")

    st.divider()

    st.markdown("### 📝 Text Analysis")

    st.write(
        "Analyze Arabic text and detect "
        "Positive or Negative sentiment."
    )

    st.divider()

    st.markdown("### 🎙️ Media Analysis")

    st.write(
        "Upload Audio or Video and automatically "
        "extract Arabic speech."
    )

    st.divider()

    st.caption("Powered by Deep Learning + Whisper")


# =========================================================
# MAIN HEADER
# =========================================================

st.markdown(
    '<div class="main-title">🧠 Arabic Sentiment Analysis</div>',
    unsafe_allow_html=True
)


st.markdown(
    '''
    <div class="subtitle">
    Analyze Arabic text, audio, and video using Deep Learning and Speech Recognition.
    </div>
    ''',
    unsafe_allow_html=True
)


# =========================================================
# CREATE TABS
# =========================================================

tab1, tab2 = st.tabs([

    "📝 Text Input Analysis",

    "🎙️ Audio / Video Analysis"

])


# =========================================================
# TAB 1: TEXT ANALYSIS
# =========================================================

with tab1:

    st.markdown(
        '<div class="section-title">Text Sentiment Analysis</div>',
        unsafe_allow_html=True
    )


    user_text = st.text_area(

        "Enter Arabic text to analyze:",

        placeholder="اكتب النص العربي هنا...",

        height=160

    )


    if st.button(
        "🔍 Analyze Text",
        key="analyze_text"
    ):


        if not user_text.strip():

            st.warning(
                "⚠️ Please enter Arabic text first."
            )


        else:

            with st.spinner(
                "Analyzing sentiment..."
            ):

                result = predict_sentiment(
                    user_text,
                    model
                )


            display_result(result)


            with st.expander(
                "📝 View Processed Text"
            ):

                st.write(
                    result["cleaned_text"]
                )


# =========================================================
# TAB 2: AUDIO / VIDEO ANALYSIS
# =========================================================

with tab2:

    st.markdown(
        '<div class="section-title">Audio / Video Sentiment Analysis</div>',
        unsafe_allow_html=True
    )


    uploaded_file = st.file_uploader(

        "Upload Audio or Video",

        type=[
            "mp3",
            "wav",
            "m4a",
            "mp4",
            "mov",
            "avi"
        ]

    )


    if uploaded_file is not None:


        file_extension = uploaded_file.name.split(".")[-1].lower()


        # Preview Media

        if file_extension in [

            "mp4",
            "mov",
            "avi"

        ]:

            st.video(uploaded_file)

        else:

            st.audio(uploaded_file)


        st.write(
            f"📁 File: **{uploaded_file.name}**"
        )


        if st.button(
            "🎙️ Analyze Media",
            key="analyze_media"
        ):


            # Create temporary file

            with tempfile.NamedTemporaryFile(

                delete=False,

                suffix=f".{file_extension}"

            ) as temp_file:


                temp_file.write(
                    uploaded_file.getbuffer()
                )


                temp_path = temp_file.name


            try:


                # Whisper

                with st.spinner(

                    "🎙️ Whisper is extracting Arabic speech..."

                ):

                    extracted_text = transcribe_media(
                        temp_path
                    )


                st.markdown(
                    '<div class="section-title">📝 Extracted Text</div>',
                    unsafe_allow_html=True
                )


                st.info(
                    extracted_text
                )


                # Sentiment

                with st.spinner(

                    "🧠 Analyzing sentiment..."

                ):

                    result = predict_sentiment(

                        extracted_text,

                        model

                    )


                display_result(result)


            except Exception as e:

                st.error(
                    f"❌ Error: {str(e)}"
                )


            finally:


                if os.path.exists(temp_path):

                    os.remove(temp_path)

Writing /kaggle/working/app.py


In [4]:
import subprocess
import time

process = subprocess.Popen([
    "streamlit",
    "run",
    "/kaggle/working/app.py",
    "--server.port=8501",
    "--server.address=0.0.0.0"
])

time.sleep(15)

print("✅ Streamlit Started")

2026-09-03 12:13:37.293 Uvicorn server started on 0.0.0.0:8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.59.215.84:8501

✅ Streamlit Started


In [6]:
from pyngrok import ngrok

ngrok.set_auth_token(
    "3IkLt95gO8fcTrcLAB1yOC3G35Q_3LWccmSBMYAbG8TfrvKZj"
)
    

In [7]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8501)

print("🚀 YOUR STREAMLIT APP:")
print(public_url)

🚀 YOUR STREAMLIT APP:
NgrokTunnel: "https://watch-blurb-scoured.ngrok-free.dev" -> "http://localhost:8501"


2026-09-03 12:14:47.457774: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
